## 1. Загрузка данных и проверка схемы

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from metric import precision_at_recall
import warnings
warnings.filterwarnings('ignore')

print("Загрузка данных...")
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(f"Train size: {train.shape}, Test size: {test.shape}, Events size: {events.shape}")

Загрузка данных...


Train size: (11091, 5), Test size: (4909, 4), Events size: (328905, 14)


## 2. Очистка данных

In [1]:
# Привязка событий к окну
meta = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']], 
    test[['cookie_id', 'window_start_ts', 'window_end_ts']]
])

events = events.merge(meta, on='cookie_id', how='left')

# Исключение событий из будущего и прошлого (строго внутри [start, end))
mask_in_window = (events['event_ts'] >= events['window_start_ts']) & (events['event_ts'] < events['window_end_ts'])
events = events[mask_in_window].copy()

# Удаление полных дублей строк событий (технические повторы записи)
events.drop_duplicates(subset=events.columns.difference(['window_start_ts', 'window_end_ts']), inplace=True)

# Сортировка по времени для корректного расчета временных интервалов
events.sort_values(['cookie_id', 'event_ts'], inplace=True)

events.drop(columns=['window_start_ts', 'window_end_ts'], inplace=True)
print(f"Events size after cleaning: {events.shape}")

NameError: name 'pd' is not defined

## 3. Генерация признаков (Feature Engineering)

Мы генерируем порядка 25 объяснимых признаков:
- Объём и разнообразие (кол-во событий, уник. объявлений, запросов).
- Признаки времени (медианный интервал, стандартное отклонение, быстрые клики < 5 сек).
- Возраст куки.
- Доли различных типов действий.

In [3]:
print("Расчет временных признаков...")
events['prev_ts'] = events.groupby('cookie_id')['event_ts'].shift(1)
events['time_diff'] = (events['event_ts'] - events['prev_ts']).dt.total_seconds()

# Флаг быстрого действия (< 5 секунд)
events['is_fast_action'] = (events['time_diff'] < 5).astype(float)

print("Агрегация признаков...")
# One-hot encoding для event_name (чтобы посчитать доли)
event_dummies = pd.get_dummies(events['event_name'], prefix='ev')
events = pd.concat([events, event_dummies], axis=1)
event_cols = event_dummies.columns.tolist()

agg_funcs = {
    'event_ts': ['min', 'max', 'count'],
    'item_id': ['nunique'],
    'item_category': ['nunique'],
    'item_location': ['nunique'],
    'search_query': ['nunique'],
    'search_page': ['max'],
    'time_diff': ['mean', 'median', 'std', 'max'],
    'is_fast_action': ['mean']
}

for c in event_cols:
    agg_funcs[c] = ['sum']
    
g = events.groupby('cookie_id')
f = g.agg(agg_funcs)

# Плоские названия колонок
f.columns = ['_'.join(col).strip() for col in f.columns.values]
f.reset_index(inplace=True)

# Производные признаки
f['session_duration'] = (f['event_ts_max'] - f['event_ts_min']).dt.total_seconds()
f['time_diff_std_to_mean'] = f['time_diff_std'] / (f['time_diff_mean'] + 1e-5)

f.drop(columns=['event_ts_min', 'event_ts_max'], inplace=True)

print("Слияние с метаданными...")
X_train = train.merge(f, on='cookie_id', how='left')
X_test = test.merge(f, on='cookie_id', how='left')

# Возраст куки на момент конца окна
X_train['cookie_age_days'] = (X_train['window_end_ts'] - X_train['cookie_created_at']).dt.total_seconds() / 86400.0
X_test['cookie_age_days'] = (X_test['window_end_ts'] - X_test['cookie_created_at']).dt.total_seconds() / 86400.0

drop_cols = ['cookie_id', 'target', 'window_start_ts', 'window_end_ts', 'cookie_created_at']
features = [c for c in X_train.columns if c not in drop_cols]
print(f"Итоговое количество признаков: {len(features)}")

Расчет временных признаков...
Агрегация признаков...


Слияние с метаданными...
Итоговое количество признаков: 23


## 4. Сравнение по датам и обучение модели

Для объективной оценки используем разбиения по датам. Обучаем LightGBM (без сложных подборов параметров, он отлично работает из коробки).

In [4]:
dates_splits = [
    ('2026-04-12', '2026-04-15', 'Split 1 (12-14 Apr)'),
    ('2026-04-15', '2026-04-17', 'Split 2 (15-16 Apr)'),
    ('2026-04-17', '2026-04-20', 'Split 3 (17-19 Apr)')
]

lgb_params = {
    'objective': 'binary',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1
}

y_train = X_train['target']
best_iterations = []

for start_date, end_date, split_name in dates_splits:
    val_mask = (X_train['window_start_ts'] >= start_date) & (X_train['window_start_ts'] < end_date)
    train_mask = X_train['window_start_ts'] < start_date
    
    tr_x, va_x = X_train[train_mask][features], X_train[val_mask][features]
    tr_y, va_y = y_train[train_mask], y_train[val_mask]
    
    model = lgb.LGBMClassifier(**lgb_params, n_estimators=500)
    model.fit(
        tr_x, tr_y,
        eval_set=[(va_x, va_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    p_va = model.predict_proba(va_x)[:, 1]
    score = precision_at_recall(va_y.values, p_va)
    print(f"{split_name} -> P@R>=0.7: {score:.4f}")
    best_iterations.append(model.best_iteration_)


Split 1 (12-14 Apr) -> P@R>=0.7: 0.3935
Split 2 (15-16 Apr) -> P@R>=0.7: 0.4074


Split 3 (17-19 Apr) -> P@R>=0.7: 0.4609


## 5. Итоговая модель и сохранение ответа

Обучаемся на всем train с усредненным количеством деревьев, предсказываем для test и применяем проверки формата (assert).

In [5]:
mean_iters = int(np.mean(best_iterations))
print(f"Обучение финальной модели на {mean_iters} итерациях...")

full_model = lgb.LGBMClassifier(**lgb_params, n_estimators=mean_iters)
full_model.fit(X_train[features], y_train)

p_test = full_model.predict_proba(X_test[features])[:, 1]

sub = pd.DataFrame({
    'cookie_id': X_test['cookie_id'],
    'score': p_test
})

# Безопасные проверки формата файла
assert len(sub) == len(test), "Неверное количество строк!"
assert set(sub['cookie_id']) == set(test['cookie_id']), "Множество cookie_id не совпадает с test.csv!"
assert sub['score'].isna().sum() == 0, "Есть пропуски в предсказаниях!"
assert sub['score'].between(0, 1).all(), "Вероятности вне диапазона [0, 1]!"

sub.to_csv('submission.csv', index=False)
print("Файл submission.csv успешно сохранен и прошел все проверки.")

Обучение финальной модели на 121 итерациях...
Файл submission.csv успешно сохранен и прошел все проверки.
